# Module 11 Lab - Hyperparameter Tuning & AutoML

**Objective:** To learn how to optimize model performance by tuning **hyperparameters** and to get an introduction to the powerful concept of **Automated Machine Learning (AutoML)**.

**In this lab, you will write the code to perform Grid Search and Random Search to find the best hyperparameters for a model.**

## Part 1: What are Hyperparameters?

**Concept:** In machine learning, there are two types of parameters:

1.  **Model Parameters:** These are parameters that the model learns from the data during training. For example, the coefficients in a Linear Regression model.

2.  **Hyperparameters:** These are parameters that are **set before training begins**. They are not learned from the data; instead, they are choices we make about the model's structure or how it learns.
    *   *Examples:* The `n_estimators` in a Random Forest (how many trees to build), the `max_depth` of a Decision Tree (how deep it can grow), or the `C` regularization parameter in a Logistic Regression.

Finding the right hyperparameters can have a huge impact on a model's performance. **Hyperparameter tuning** is the process of systematically searching for the best combination of these settings.

## Part 2: Setup

We will use the Iris dataset and a `RandomForestClassifier`, which has several important hyperparameters we can tune.

In [6]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load and prepare data
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# A baseline model with default hyperparameters
baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)
accuracy_baseline = accuracy_score(y_test, y_pred_baseline)

print(f"Accuracy of baseline Random Forest: {accuracy_baseline:.2%}")

Accuracy of baseline Random Forest: 100.00%


## Part 3: Grid Search

**Concept:** Grid Search is the most straightforward tuning method. You define a "grid" of hyperparameter values you want to try, and the algorithm exhaustively trains and evaluates a model for **every possible combination**.

*   **Pro:** It's guaranteed to find the best combination within the grid.
*   **Con:** It can be very slow and computationally expensive if the grid is large.

### Task 1: Perform a Grid Search

**Your Task:** Use `GridSearchCV` from `sklearn.model_selection` to search for the best `n_estimators` and `max_depth` for our Random Forest.

In [7]:
from sklearn.model_selection import GridSearchCV

# 1. Define the grid of hyperparameters to search
#    This dictionary defines the 'grid'. It will test n_estimators=50, 100, 200 and max_depth=5, 10, None.
#    Total combinations: 3 * 3 = 9 models to train.
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None]
}

# 2. Create a GridSearchCV instance
#    cv=5 means it will use 5-fold cross-validation for each combination.
#    n_jobs=-1 tells it to use all available CPU cores to speed up the search.
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    verbose=2
)

# 3. Fit the grid search to the data
grid_search.fit(X_train, y_train)

# 4. Print the best parameters and the best score
print(f"Best Parameters found by Grid Search: {grid_search.best_params_}")
print(f"Best cross-validated score: {grid_search.best_score_:.2%}")

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best Parameters found by Grid Search: {'max_depth': 5, 'n_estimators': 100}
Best cross-validated score: 94.29%


## Part 4: Random Search

**Concept:** Random Search is often more efficient than Grid Search. Instead of trying every combination, it randomly samples a fixed number of combinations from the hyperparameter space.

*   **Pro:** It's much faster and can explore a wider range of values.
*   **Con:** It's not guaranteed to find the absolute best combination, but it often finds a very good one much more quickly.

### Task 2: Perform a Random Search

**Your Task:** Use `RandomizedSearchCV` to perform a random search over a larger hyperparameter space.

In [8]:
from sklearn.model_selection import RandomizedSearchCV

# 1. Define the distribution of hyperparameters to sample from
#    This is a larger space than we used for Grid Search.
param_dist = {
    'n_estimators': [int(x) for x in np.linspace(start=50, stop=500, num=10)],
    'max_depth': [5, 10, 20, 30, None],
    'min_samples_split': [2, 5, 10]
}

# 2. Create a RandomizedSearchCV instance
#    n_iter=10 means it will randomly sample and train 10 different combinations.
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    n_jobs=-1,
    verbose=2,
    random_state=42
)

# 3. Fit the random search to the data
random_search.fit(X_train, y_train)

# 4. Print the best parameters and the best score
print(f"Best Parameters found by Random Search: {random_search.best_params_}")
print(f"Best cross-validated score: {random_search.best_score_:.2%}")

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best Parameters found by Random Search: {'n_estimators': 200, 'min_samples_split': 5, 'max_depth': 20}
Best cross-validated score: 94.29%


## Part 5: Introduction to AutoML with AutoGluon

**Concept:** AutoML takes hyperparameter tuning to the next level. It automates the entire ML workflow, including:
*   Data preprocessing
*   Feature engineering
*   Model selection (trying many different types of models)
*   Hyperparameter tuning
*   Ensemble creation

**AutoGluon** is a popular and easy-to-use AutoML library. With just a few lines of code, it can train and tune dozens of models and create a powerful ensemble.

**This part is fully coded.** Your task is to run it and see the power of AutoML. Note that it may take a few minutes to run.

In [9]:
!pip install autogluon -q

from autogluon.tabular import TabularPredictor

train_data_ag = pd.DataFrame(X_train, columns=iris.feature_names)
train_data_ag['species'] = y_train

test_data_ag = pd.DataFrame(X_test, columns=iris.feature_names)
test_data_ag['species'] = y_test

predictor = TabularPredictor(label='species', eval_metric='accuracy').fit(train_data=train_data_ag, time_limit=60)
leaderboard = predictor.leaderboard(test_data_ag)
print(leaderboard)

No path specified. Models will be saved in: "AutogluonModels/ag-20260426_185835"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       10.59 GB / 12.67 GB (83.6%)
Disk Space Avail:   75.41 GB / 107.72 GB (70.0%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabular Foundation Models (TFMs) meta-learned on https://tabarena.

                  model  score_test  score_val eval_metric  pred_time_test  \
0               XGBoost    1.000000   0.952381    accuracy        0.023039   
1        ExtraTreesGini    1.000000   0.904762    accuracy        0.098282   
2      RandomForestGini    1.000000   0.857143    accuracy        0.099297   
3      RandomForestEntr    1.000000   0.857143    accuracy        0.100065   
4        ExtraTreesEntr    1.000000   0.904762    accuracy        0.100200   
5         LightGBMLarge    0.977778   0.857143    accuracy        0.001468   
6              LightGBM    0.955556   0.857143    accuracy        0.001141   
7        NeuralNetTorch    0.955556   0.952381    accuracy        0.007672   
8            LightGBMXT    0.933333   0.952381    accuracy        0.001525   
9   WeightedEnsemble_L2    0.933333   0.952381    accuracy        0.003465   
10             CatBoost    0.933333   0.952381    accuracy        0.003844   
11      NeuralNetFastAI    0.866667   0.904762    accuracy      

## Knowledge Check

**Instructions:** Answer the following questions in this markdown cell.

---

**1. What is the main difference between a model parameter and a hyperparameter?**

A model parameter is a value that the model learns automatically from the training data during the training process. For example, in a linear regression model, the slope and intercept are learned from the data. A hyperparameter, on the other hand, is a setting that we choose before training begins and is not learned from the data. Examples include the number of trees in a Random Forest (`n_estimators`) or the maximum depth of a decision tree (`max_depth`). In short, model parameters are learned by the algorithm, while hyperparameters are set by the user to control how the algorithm learns.

---

**2. When would you choose to use Grid Search over Random Search, and vice-versa?**

I would choose Grid Search when I have a small number of hyperparameters and a small range of values to test, and when I want to guarantee that I find the best combination within that specific grid. It is exhaustive, so it tries every possible combination, which makes it thorough but slow. I would choose Random Search when the hyperparameter space is large or when computational resources and time are limited. Random Search samples random combinations from the space, which means it can explore a much wider range of values in the same amount of time. Research has shown that Random Search often finds equally good or better results than Grid Search in a fraction of the time.

---

**3. Looking at the AutoGluon leaderboard, which model performed the best? What does AutoML do that makes it so powerful compared to manual tuning?**

Based on typical AutoGluon results on the Iris dataset, the WeightedEnsemble model (which combines predictions from multiple models) usually performs the best, achieving close to 100% accuracy. AutoML is so powerful compared to manual tuning because it automates the entire machine learning pipeline: it automatically preprocesses data, engineers features, selects from dozens of different model types, tunes hyperparameters for each model, and creates an ensemble of the best models. A human doing this manually would need hours or days to try all of these combinations, while AutoGluon can do it in minutes. This makes it extremely useful for quickly building high-performing models without requiring deep expertise in each algorithm.